In [1]:
from pathlib import Path

print("All PDB files here:", [f.name for f in Path(".").glob("*.pdb")])

All PDB files here: ['aligned_1m17.pdb', '4hjo.pdb', 'aligned_ligand.pdb', '1M17.pdb', 'taligned_ligand.pdb']


In [2]:
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw

# Find the saved ligand PDB files from the previous step
# (Update the path if you saved them in a specific directory)
mol_files = sorted(list(Path(".").glob("*_ligand.pdb")))
print("Found ligand files:", [f.name for f in mol_files])

# Read them into RDKit molecule objects
molecules = []
for mol_file in mol_files:
    molecule = Chem.MolFromPDBFile(str(mol_file), removeHs=False)
    if molecule is None:
        print(f"{mol_file.name} could not be read")
    else:
        Chem.SanitizeMol(molecule)
        molecules.append(molecule)

print(f"Successfully loaded {len(molecules)} molecules into RDKit!")

Found ligand files: ['aligned_ligand.pdb', 'taligned_ligand.pdb']
Successfully loaded 2 molecules into RDKit!


In [3]:
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
import os

# Load the default feature definition factory from RDKit
fdef_path = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
factory = ChemicalFeatures.BuildFeatureFactory(fdef_path)

# Extract features for your molecule(s)
all_features = []
for mol in molecules:
    features = factory.GetFeaturesForMol(mol)
    all_features.append(features)

print(f"Extracted feature sets for {len(molecules)} molecule(s).")
for i, feats in enumerate(all_features):
    print(f"Molecule {i+1} has {len(feats)} total pharmacophoric features.")

Extracted feature sets for 2 molecule(s).
Molecule 1 has 13 total pharmacophoric features.
Molecule 2 has 13 total pharmacophoric features.


In [4]:
for i, feats in enumerate(all_features):
    print(f"\n--- Features for Molecule {i+1} ---")
    for feat in feats:
        print(f"Family: {feat.GetFamily()} | Type: {feat.GetType()} | Position: ({feat.GetPos().x:.2f}, {feat.GetPos().y:.2f}, {feat.GetPos().z:.2f})")


--- Features for Molecule 1 ---
Family: Donor | Type: SingleAtomDonor | Position: (25.10, 12.10, -1.64)
Family: Donor | Type: SingleAtomDonor | Position: (27.21, 8.46, -0.80)
Family: Donor | Type: SingleAtomDonor | Position: (27.20, 10.83, -1.40)
Family: Acceptor | Type: SingleAtomAcceptor | Position: (21.48, 8.54, -0.67)
Family: Acceptor | Type: SingleAtomAcceptor | Position: (18.63, 8.76, -1.34)
Family: Acceptor | Type: SingleAtomAcceptor | Position: (22.95, 6.02, -0.07)
Family: Acceptor | Type: SingleAtomAcceptor | Position: (22.02, 4.30, 2.12)
Family: PosIonizable | Type: BasicGroup | Position: (25.10, 12.10, -1.64)
Family: PosIonizable | Type: BasicGroup | Position: (27.21, 8.46, -0.80)
Family: PosIonizable | Type: BasicGroup | Position: (27.20, 10.83, -1.40)
Family: Hydrophobe | Type: ThreeWayAttach | Position: (26.95, 14.76, -3.54)
Family: Hydrophobe | Type: ThreeWayAttach | Position: (25.07, 9.68, -1.04)
Family: Hydrophobe | Type: ChainTwoWayAttach | Position: (27.93, 14.81, -

In [5]:
# Collect coordinates for each feature type across all loaded molecules
feature_coords = {
    "donors": [],
    "acceptors": [],
    "hydrophobics": []
}

for feats in all_features:
    for feat in feats:
        family = feat.GetFamily()
        pos = feat.GetPos()
        coord = [pos.x, pos.y, pos.z]
        
        if family == "Donor":
            feature_coords["donors"].append(coord)
        elif family == "Acceptor":
            feature_coords["acceptors"].append(coord)
        elif family == "Hydrophobe":
            feature_coords["hydrophobics"].append(coord)

# Print out summary counts
for f_type, coords in feature_coords.items():
    print(f"Total {f_type} collected: {len(coords)}")

Total donors collected: 6
Total acceptors collected: 8
Total hydrophobics collected: 6


In [6]:
import numpy as np
from scipy.cluster.hierarchy import fclusterdata

# Define a distance threshold for clustering (e.g., features within 2.0 Angstroms merge)
distance_threshold = 2.0

consensus_pharmacophore = {}

for f_type, coords in feature_coords.items():
    if len(coords) == 0:
        continue
    
    coords_array = np.array(coords)
    
    # Perform hierarchical clustering on the coordinates
    # fclusterdata clusters data based on a maximum distance criterion
    clusters = fclusterdata(coords_array, t=distance_threshold, criterion='distance')
    
    # Calculate the centroid (average position) for each cluster
    cluster_centers = []
    for cluster_id in np.unique(clusters):
        cluster_points = coords_array[clusters == cluster_id]
        centroid = cluster_points.mean(axis=0)
        cluster_centers.append(centroid)
        
    consensus_pharmacophore[f_type] = cluster_centers
    print(f"Consensus {f_type}: {len(cluster_centers)} point(s) generated from {len(coords)} raw features.")

Consensus donors: 3 point(s) generated from 6 raw features.
Consensus acceptors: 4 point(s) generated from 8 raw features.
Consensus hydrophobics: 2 point(s) generated from 6 raw features.


In [7]:
# Define cluster selection criteria from TeachOpenCADD T009
min_cluster_size = 2  # Minimum number of features required in a cluster
top_cluster_number = 3 # Maximum number of top clusters to keep per type

def get_clusters(cluster_results, min_size, top_n):
    # cluster_results comes from fclusterdata
    unique_clusters, counts = np.unique(cluster_results, return_counts=True)
    # Sort clusters by size (descending)
    sorted_indices = np.argsort(-counts)
    
    valid_clusters = []
    for idx in sorted_indices:
        if counts[idx] >= min_size and len(valid_clusters) < top_n:
            valid_clusters.append(unique_clusters[idx])
    return valid_clusters

# Filter each feature type
selected_pharmacophore = {}
for f_type, coords in feature_coords.items():
    if len(coords) == 0:
        continue
    coords_array = np.array(coords)
    clusters = fclusterdata(coords_array, t=2.0, criterion='distance')
    
    valid_cluster_ids = get_clusters(clusters, min_cluster_size, top_cluster_number)
    
    centers = []
    for cid in valid_cluster_ids:
        cluster_points = coords_array[clusters == cid]
        centers.append(cluster_points.mean(axis=0))
        
    selected_pharmacophore[f_type] = centers
    print(f"Filtered {f_type}: kept {len(centers)} robust consensus point(s).")

Filtered donors: kept 3 robust consensus point(s).
Filtered acceptors: kept 2 robust consensus point(s).
Filtered hydrophobics: kept 2 robust consensus point(s).


In [11]:
!pip install nglview

In [16]:
# Generate a PyMOL script (.pml) to draw your pharmacophore points as spheres
pml_script = """
# Clear any existing pseudoatoms
delete pharmacophore_*

"""

# Color mapping for feature types
colors = {
    "donors": "blue",       # Donors in blue
    "acceptors": "red",     # Acceptors in red
    "hydrophobics": "yellow"# Hydrophobics in yellow
}

for f_type, centers in selected_pharmacophore.items():
    color = colors.get(f_type, "green")
    for idx, center in enumerate(centers):
        name = f"{f_type}_{idx+1}"
        x, y, z = center
        pml_script += f"pseudoatom pharmacophore_{f_type}, pos=[{x:.3f}, {y:.3f}, {z:.3f}], name={name}, color={color}\n"

# Add representation commands
pml_script += """
show spheres, pharmacophore_*
set sphere_scale, 0.4, pharmacophore_*
"""

# Save to file
with open("pharmacophore_points.pml", "w") as f:
    f.write(pml_script)

print("Saved 'pharmacophore_points.pml'! Open PyMOL and run: @pharmacophore_points.pml")

Saved 'pharmacophore_points.pml'! Open PyMOL and run: @pharmacophore_points.pml
